In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone per rule 10
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Attach timezone-aware timestamps (UTC+8) as a new column (do not modify original numeric timestamp)
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s').dt.tz_localize(tz)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s').dt.tz_localize(tz)
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s').dt.tz_localize(tz)
df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s').dt.tz_localize(tz)

# Helper to produce grouped summary for a given dataframe and name column
def grouped_summary(df, name_col):
    g = df.groupby(['cmdb_id', name_col], as_index=False).agg(
        count=('value', 'count'),
        min=('value', 'min'),
        P50=('value', lambda x: x.quantile(0.5)),
        P95=('value', lambda x: x.quantile(0.95)),
        max=('value', 'max')
    )
    # Ensure count is int
    g['count'] = g['count'].astype(int)
    # Sort by P95 descending and keep top 20
    g_sorted = g.sort_values('P95', ascending=False).reset_index(drop=True).head(20)
    return g, g_sorted  # return full summary (in memory) and compact table

# Compute summaries
metric_full_summary, metric_top20 = grouped_summary(df_metric, 'kpi_name')
trace_full_summary, trace_top20 = grouped_summary(df_trace, 'trace_name')
log_full_summary, log_top20 = grouped_summary(df_log, 'log_name')

# Unique cmdb_id values across all files
unique_cmdb_ids = sorted(set(df_metric['cmdb_id'].dropna().unique()) |
                         set(df_trace['cmdb_id'].dropna().unique()) |
                         set(df_log['cmdb_id'].dropna().unique()) |
                         set(df_err['cmdb_id'].dropna().unique()))

# Error logs row count
error_logs_count = df_err.shape[0]

# Compact outputs (these variables will be displayed)
metric_top20, trace_top20, log_top20, unique_cmdb_ids, error_logs_count

```
Out[1]:
```
summary = (
    "Global summary (computed across all data in metric.csv, trace.csv, log.csv; timestamps UTC+8):\n\n"
    "Metrics: The highest P95 values are observed on Redis and Tomcat JVM metrics. Redis (Redis01/Redis02) "
    "shows extremely large P95s for network packets and Redis counters (e.g., P95 values on the order of 1e9–5.7e9) and maxmemory ~4.294967e9. "
    "Tomcat instances show large JVM memory metrics (JVM Total/Max ~4.1876e9; JVM Free ~2.7–3.1e9). Container memory entries around 3.072e9 appear as well.\n\n"
    "Traces: The top P95 trace groups are trace.self.row_count for dockerB2 (P95 ~2270), dockerB1 (P95 ~2204), and Tomcat nodes (P95 ~1.7k). "
    "There are also per-source/target row_count traces between containers and MG/IG components with lower P95s (~100–160).\n\n"
    "Logs: The highest P95 log groups are log.row_count for apache02 (P95 ~736) and apache01 (P95 ~503), with Tomcat log.row_count and small GC pause maxima on Tomcats. log.error_count is zero across Tomcats and Apache in the sampled data.\n\n"
    "Components: Found {} unique cmdb_id values (examples: {}).\n"
    "Error logs: error_logs.csv contains {} rows.".format(
        len(unique_cmdb_ids),
        ", ".join(unique_cmdb_ids[:8]) + (" ..." if len(unique_cmdb_ids) > 8 else ""),
        error_logs_count
    )
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count           min           P50           P95           max
0    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15  1.744000e+03  2.105000e+03  5.697440e+09  5.697763e+09
1    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     15  1.594000e+03  1.942000e+03  5.169741e+09  5.170038e+09
2    Redis01  redis-Redis_6379_Redis  (total_commands_proces...     29  5.030803e+09  5.032299e+09  5.033726e+09  5.033883e+09
3    Redis01                redis-Redis_6379_Redis  (maxmemory)     29  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
4    Redis02                redis-Redis_6379_Redis  (maxmemory)     29  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
5   Tomcat02           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
6   Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
7   Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
8   Tomcat04           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
9   Tomcat01             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
10  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
11  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
12  Tomcat04             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09  4.187619e+09  4.187619e+09  4.187619e+09
13   Redis02  redis-Redis_6379_Redis  (total_commands_proces...     29  3.998384e+09  3.999408e+09  4.000320e+09  4.000412e+09
14   Redis01            redis-Redis_6379_Redis  (keyspace_hits)     29  3.823231e+09  3.824378e+09  3.825477e+09  3.825598e+09
15  Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4  2.862059e+09  3.080380e+09  3.160090e+09  3.173950e+09
16  dockerB2  Container-DOCKER_CONTAINER_1bc4fc80d241--bcou-...      1  3.072000e+09  3.072000e+09  3.072000e+09  3.072000e+09
17  dockerA1  Container-DOCKER_CONTAINER_bbc780df1fce--bcou-...      2  3.072000e+09  3.072000e+09  3.072000e+09  3.072000e+09
18  Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      6  2.834152e+09  2.908522e+09  2.993438e+09  3.013509e+09
19  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory      4  2.693680e+09  2.783205e+09  2.917849e+09  2.927303e+09,      cmdb_id                     trace_name  count     min     P50      P95     max
0   dockerB2           trace.self.row_count     30   891.0  1447.5  2270.10  2504.0
1   dockerB1           trace.self.row_count     30   808.0  1467.5  2203.55  2403.0
2   Tomcat03           trace.self.row_count     30  1109.0  1437.0  1769.10  1780.0
3   Tomcat02           trace.self.row_count     30  1210.0  1406.5  1739.60  1760.0
4   Tomcat01           trace.self.row_count     30  1172.0  1504.0  1738.10  1847.0
5   Tomcat04           trace.self.row_count     30  1060.0  1453.0  1732.30  1752.0
6   dockerA2           trace.self.row_count     30   608.0  1071.0  1590.65  1642.0
7   dockerA1           trace.self.row_count     30   623.0  1000.5  1518.60  1568.0
8       IG02           trace.self.row_count     30   700.0   862.0  1005.10  1092.0
9       IG01           trace.self.row_count     30   740.0   837.0  1003.50  1030.0
10      MG01           trace.self.row_count     30   509.0   648.5   809.85   839.0
11      MG02           trace.self.row_count     30   507.0   664.5   760.85   827.0
12      MG01  trace.from_dockerB2.row_count     30    36.0    90.5   155.95   168.0
13  dockerB2        trace.to_MG01.row_count     30    36.0    90.5   155.95   168.0
14      MG01  trace.from_dockerB1.row_count     30    32.0    88.0   149.20   160.0
15  dockerB1        trace.to_MG01.row_count     30    32.0    88.0   149.20   160.0
16  dockerB1        trace.to_MG02.row_count     30    39.0    86.5   145.90   154.0
17      MG02  trace.from_dockerB1.row_count     30    39.0    86.5   145.90   154.0
18      IG02    trace.to_Tomcat04.row_count     30    78.0   105.0   140.50   147.0
19  Tomcat04      trace.from_IG02.row_count     30    78.0   105.0   140.50   147.0,      cmdb_id          log_name  count    min      P50      P95     max
0   apache02     log.row_count     31  15.00  445.000  736.000  747.00
1   apache01     log.row_count     31   3.00  411.000  503.500  522.00
2   Tomcat03     log.row_count     31   3.00  206.000  258.500  265.00
3   Tomcat01     log.row_count     31   9.00  218.000  257.500  264.00
4   Tomcat04     log.row_count     31   2.00  215.000  254.500  259.00
5   Tomcat02     log.row_count     19   1.00    1.000    1.000    1.00
6   Tomcat01  log.gc_pause_max     12   0.03    0.075    0.099    0.11
7   Tomcat02  log.gc_pause_max     19   0.02    0.060    0.081    0.09
8   Tomcat04  log.gc_pause_max     12   0.03    0.065    0.080    0.08
9   Tomcat03  log.gc_pause_max     12   0.02    0.055    0.080    0.08
10  Tomcat01   log.error_count     31   0.00    0.000    0.000    0.00
11  Tomcat02   log.error_count     19   0.00    0.000    0.000    0.00
12  Tomcat03   log.error_count     31   0.00    0.000    0.000    0.00
13  Tomcat04   log.error_count     31   0.00    0.000    0.000    0.00
14  apache01   log.error_count     31   0.00    0.000    0.000    0.00
15  apache02   log.error_count     31   0.00    0.000    0.000    0.00, ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 0)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# timezone per rule 10
tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure we have a global metric summary named metric_summary (per instruction)
if 'metric_summary' in globals():
    metric_summary = globals()['metric_summary']
elif 'metric_full_summary' in globals():
    metric_summary = globals()['metric_full_summary']
else:
    # compute global summary (must be computed before filtering per rule 9)
    metric_summary = df_metric.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        count=('value','count'),
        min=('value','min'),
        P50=('value', lambda x: x.quantile(0.5)),
        P95=('value', lambda x: x.quantile(0.95)),
        max=('value','max')
    )
    metric_summary['count'] = metric_summary['count'].astype(int)

# Incident window in UTC+8
start_ts = int(pd.Timestamp('2021-03-09 20:00:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-09 20:30:00', tz=tz).timestamp())

# 1) Filter metric rows inside window (using numeric timestamp seconds)
metric_window_df = df_metric[df_metric['timestamp'].between(start_ts, end_ts)].copy()

# 2) Join with metric_summary to get global P95
# Ensure metric_summary has 'P95'
if 'P95' not in metric_summary.columns:
    # compute P95 if not present
    metric_summary = metric_summary.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        count=('count','first'),
        min=('min','first'),
        P50=('P50','first'),
        P95=('P95','first'),
        max=('max','first')
    )

metric_window_merged = metric_window_df.merge(
    metric_summary[['cmdb_id','kpi_name','P95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# 3) Identify anomaly rows where value >= P95 (require P95 present)
metric_anomalies_df = metric_window_merged[metric_window_merged['P95'].notna()].copy()
metric_anomalies_df = metric_anomalies_df[metric_anomalies_df['value'] >= metric_anomalies_df['P95']].copy()
metric_anomalies_df['ratio'] = metric_anomalies_df['value'] / metric_anomalies_df['P95']

# Add human-readable timestamp in UTC+8 for compact display
metric_anomalies_df['ts_utc8'] = pd.to_datetime(metric_anomalies_df['timestamp'], unit='s').dt.tz_localize(tz)

# 4a) Top 20 anomaly rows sorted by ratio desc with selected columns
anomaly_top20 = metric_anomalies_df.sort_values('ratio', ascending=False)[
    ['cmdb_id','kpi_name','timestamp','ts_utc8','value','P95','ratio']
].head(20)

# 4b) Aggregated anomaly counts per cmdb_id: top 10 by anomaly_count
agg = metric_anomalies_df.groupby('cmdb_id', as_index=False).agg(
    anomaly_count=('ratio','count'),
    earliest_anomaly_ts_num=('timestamp','min')
)
agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_ts_num'], unit='s').dt.tz_localize(tz)
anomaly_agg_top10 = agg.sort_values('anomaly_count', ascending=False).head(10)[
    ['cmdb_id','anomaly_count','earliest_anomaly_timestamp']
]

# Store full filtered metric-window table and detected anomaly table in kernel variables (already done)
# metric_window_df, metric_anomalies_df

# Compact outputs
anomaly_top20, anomaly_agg_top10

```
Out[2]:
```
metric_anomalies_df.shape, anomaly_top20.shape, anomaly_agg_top10.shape
summary_parts = []

total_anomalies = int(metric_anomalies_df.shape[0])

# Top anomaly (first row of anomaly_top20)
if not anomaly_top20.empty:
    top = anomaly_top20.iloc[0]
    top_ratio = top['ratio']
    top_ratio_str = "inf" if (pd.isna(top_ratio) or top_ratio == float('inf')) else f"{top_ratio:.2f}"
    summary_parts.append(
        f"Total metric anomalies in the window: {total_anomalies}.\n"
        f"Top anomaly: {top['cmdb_id']} - {top['kpi_name']} at {top['ts_utc8']} value={top['value']} P95={top['P95']} ratio={top_ratio_str}."
    )
else:
    summary_parts.append(f"No anomalies detected in the window (total {total_anomalies}).")

# Highest finite ratio examples (up to 3)
finite_examples = anomaly_top20[anomaly_top20['ratio'].replace([float('inf'), float('nan')], None).notna()].copy()
finite_examples_sorted = finite_examples.sort_values('ratio', ascending=False).head(3)
if not finite_examples_sorted.empty:
    ex_lines = []
    for _, r in finite_examples_sorted.iterrows():
        ex_lines.append(f"{r['cmdb_id']}|{r['kpi_name']} value={r['value']} P95={r['P95']} ratio={r['ratio']:.2f}")
    summary_parts.append("High-ratio examples (top 3): " + "; ".join(ex_lines) + ".")

# Mention infinite-ratio cases if any
inf_count = int((anomaly_top20['ratio'] == float('inf')).sum())
if inf_count > 0:
    summary_parts.append(f"{inf_count} anomaly rows had P95==0 causing infinite ratio (e.g., a TCP-CLOSE-WAIT metric on Tomcat04).")

# Top components by anomaly counts
agg_lines = []
for _, row in anomaly_agg_top10.iterrows():
    agg_lines.append(f"{row['cmdb_id']} ({int(row['anomaly_count'])} anomalies, earliest {row['earliest_anomaly_timestamp']})")
summary_parts.append("Top components by anomaly count: " + "; ".join(agg_lines) + ".")

# Short conclusion
summary_parts.append("Summary conclusion: MySQL (Mysql01/Mysql02), Redis (Redis01/Redis02), Tomcat nodes, and MG01/IG01 show the most metric anomalies in the requested window; notable single-point spikes include TCP-CLOSE-WAIT on Tomcat04 and several MySQL metrics with ~20x P95 exceedance.")

summary = "\n\n".join(summary_parts)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name   timestamp                   ts_utc8    value        P95      ratio
20120  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1615292520 2021-03-09 12:22:00+08:00   1.0000   0.000000        inf
26862   Mysql01                        Mysql-MySQL_3306_Com Insert  1615293000 2021-03-09 12:30:00+08:00   0.2667   0.013335  20.000000
3576    Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime  1615291440 2021-03-09 12:04:00+08:00   1.0000   0.050000  20.000000
3634    Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime  1615291440 2021-03-09 12:04:00+08:00   1.0000   0.050000  20.000000
23419   Mysql02          Mysql-MySQL_3306_Opened table definitions  1615292760 2021-03-09 12:26:00+08:00   0.1667   0.008335  20.000000
14211      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1615292160 2021-03-09 12:16:00+08:00   1.0000   0.050000  20.000000
23418   Mysql02                     Mysql-MySQL_3306_Opened Tables  1615292760 2021-03-09 12:26:00+08:00   1.0333   0.130800   7.899847
23445   Mysql02           Mysql-MySQL_3306_Table open cache misses  1615292760 2021-03-09 12:26:00+08:00   1.0333   0.130800   7.899847
23362   Mysql02                  Mysql-MySQL_3306_Handler Read Rnd  1615292760 2021-03-09 12:26:00+08:00   0.2833   0.061665   4.594178
23439   Mysql02                        Mysql-MySQL_3306_Sort Range  1615292760 2021-03-09 12:26:00+08:00   0.2833   0.061665   4.594178
18441  apache01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  1615292400 2021-03-09 12:20:00+08:00   1.0000   0.250000   4.000000
23898  apache01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  1615292760 2021-03-09 12:26:00+08:00  59.0000  15.600000   3.782051
12011  Tomcat04                             OSLinux-CPU_CPU_CPUWio  1615291980 2021-03-09 12:13:00+08:00   1.8753   0.535700   3.500653
20804   Redis02                             OSLinux-CPU_CPU_CPUWio  1615292580 2021-03-09 12:23:00+08:00   0.2835   0.083309   3.403014
12423   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  1615292040 2021-03-09 12:14:00+08:00   4.0000   1.200000   3.333333
3526    Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  1615291440 2021-03-09 12:04:00+08:00   0.4000   0.120000   3.333333
19052   Redis01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  1615292460 2021-03-09 12:21:00+08:00   4.0000   1.200000   3.333333
10914   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  1615291920 2021-03-09 12:12:00+08:00   7.0000   2.100000   3.333333
112     Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead  1615291200 2021-03-09 12:00:00+08:00  22.4000   6.720000   3.333333
12431   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  1615292040 2021-03-09 12:14:00+08:00   1.0000   0.300000   3.333333,      cmdb_id  anomaly_count earliest_anomaly_timestamp
4    Mysql01            834  2021-03-09 12:00:00+08:00
5    Mysql02            751  2021-03-09 12:00:00+08:00
6    Redis01            570  2021-03-09 12:00:00+08:00
7    Redis02            548  2021-03-09 12:00:00+08:00
20  Tomcat02            545  2021-03-09 12:00:00+08:00
22  Tomcat04            515  2021-03-09 12:00:00+08:00
21  Tomcat03            488  2021-03-09 12:00:00+08:00
2       MG01            475  2021-03-09 12:00:00+08:00
19  Tomcat01            471  2021-03-09 12:00:00+08:00
0       IG01            449  2021-03-09 12:00:00+08:00)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone and df_trace if available
tz = pytz.timezone('Asia/Shanghai')

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure we have a global trace_summary with P95 and P5
if 'trace_summary' in globals():
    trace_summary = globals()['trace_summary']
else:
    # compute global summary (P5 and P95) using entire trace series per rule 9
    trace_summary = df_trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        count=('value','count'),
        min=('value','min'),
        P5=('value', lambda x: x.quantile(0.05)),
        P50=('value', lambda x: x.quantile(0.5)),
        P95=('value', lambda x: x.quantile(0.95)),
        max=('value','max')
    )
    trace_summary['count'] = trace_summary['count'].astype(int)

# Incident window in UTC+8
start_ts = int(pd.Timestamp('2021-03-09 20:00:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-09 20:30:00', tz=tz).timestamp())

# 1) Filter trace rows inside the window
trace_window_df = df_trace[df_trace['timestamp'].between(start_ts, end_ts)].copy()

# 2) Join with trace_summary to get P95 and P5
trace_window_merged = trace_window_df.merge(
    trace_summary[['cmdb_id','trace_name','P95','P5']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# 3) Identify high and low anomalies
# High: value >= P95 (handle P95==0 => ratio = inf when value>0; if both zero, ratio=1)
def compute_ratio(row):
    p95 = row['P95']
    val = row['value']
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        if val == 0:
            return 1.0
        return np.inf if val > 0 else np.nan
    return val / p95

def compute_ratio_low(row):
    p5 = row['P5']
    val = row['value']
    if pd.isna(p5):
        return np.nan
    if p5 == 0:
        # if both zero, consider ratio_low = 1 (exactly at P5); if value < 0 unlikely, else if value>0 it's not low anomaly
        if val == 0:
            return 1.0
        # value>0 cannot be <= P5(0), so this should not be a low-anomaly; keep inf to indicate non-applicable
        return np.inf
    return val / p5

trace_window_merged['ratio'] = trace_window_merged.apply(compute_ratio, axis=1)
trace_window_merged['ratio_low'] = trace_window_merged.apply(compute_ratio_low, axis=1)

# Flags for anomaly types
trace_window_merged['is_high'] = trace_window_merged['P95'].notna() & (trace_window_merged['value'] >= trace_window_merged['P95'])
trace_window_merged['is_low'] = trace_window_merged['P5'].notna() & (trace_window_merged['value'] <= trace_window_merged['P5'])

# Keep only rows that are high or low anomalies
trace_anomalies_df = trace_window_merged[trace_window_merged['is_high'] | trace_window_merged['is_low']].copy()

# human-readable timestamp in UTC+8
trace_anomalies_df['ts_utc8'] = pd.to_datetime(trace_anomalies_df['timestamp'], unit='s').dt.tz_localize(tz)

# 4A) Prepare severity sorting:
# For sorting: high anomalies first (is_high True), sorted by ratio desc.
# Then low anomalies, sorted by ratio_low asc (we invert ratio_low to make descending severity metric).
trace_anomalies_df['severity_group'] = np.where(trace_anomalies_df['is_high'], 1, 0)
# severity_metric: for highs use ratio, for lows use -ratio_low (so larger severity_metric is more severe in both cases)
trace_anomalies_df['severity_metric'] = np.where(
    trace_anomalies_df['is_high'],
    trace_anomalies_df['ratio'].replace([np.nan, np.inf], -np.inf),      # prefer finite ratios; inf stays large
    -trace_anomalies_df['ratio_low'].replace([np.nan, np.inf], np.inf)   # for lows, smaller ratio_low => larger -ratio_low
)

anomaly_top20 = trace_anomalies_df.sort_values(
    by=['severity_group','severity_metric'],
    ascending=[False, False]
)[['cmdb_id','trace_name','timestamp','ts_utc8','value','P95','P5','ratio','ratio_low']].head(20)

# 4B) Aggregated anomaly counts per cmdb_id
agg = trace_anomalies_df.groupby('cmdb_id', as_index=False).agg(
    high_anomaly_count=('is_high','sum'),
    low_anomaly_count=('is_low','sum'),
    earliest_anomaly_ts_num=('timestamp','min')
)
agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_ts_num'], unit='s').dt.tz_localize(tz)
agg['total_anomalies'] = agg['high_anomaly_count'] + agg['low_anomaly_count']
anomaly_agg_top10 = agg.sort_values('total_anomalies', ascending=False).head(10)[
    ['cmdb_id','high_anomaly_count','low_anomaly_count','earliest_anomaly_timestamp']
]

# Store full filtered trace-window table and detected trace anomaly table in kernel variables
# (trace_window_df already defined; trace_anomalies_df defined)

# Compact outputs
anomaly_top20, anomaly_agg_top10

```
Out[3]:
```
Summary of trace anomaly detection within the requested window (results shown with timestamps in UTC+8):

- Many trace anomalies were detected, dominated by duration-related metrics (duration_p95) and some row_count spikes.
- Highest-severity examples: 
  - MG01 trace.from_dockerB1.duration_p95 (value 0.8864 vs P95 0.2025) — ratio ≈ 4.38.
  - dockerB1 trace.to_MG01.duration_p95 (same values) — ratio ≈ 4.38.
  - dockerA2 trace.self.duration_p95 (value 1.0048 vs P95 0.30198) — ratio ≈ 3.33.
  - Several MG02 / dockerA2 / dockerB2 traces also show ratios between ~1.6–1.94.
- Some records show very large value/P5 numbers because P5 is very small (P5 near 0), which amplifies ratio_low; the top anomalies list includes both large high-side exceedances and large ratio_low numbers due to tiny P5.
- Aggregated by component (top 10 by total anomalies): MG01 and MG02 have the most anomalies (MG01: 81 high, 94 low; MG02: 78 high, 94 low), followed by dockerB1, dockerA1, dockerB2, Tomcat04, Tomcat02, dockerA2, Tomcat01, and IG01. Earliest detected anomalies for many components are shown at 2021-03-09 12:00:00+08 in the results.
- Conclusion / next steps: the trace anomalies point to latency spikes and abnormal durations concentrated around MG01/MG02 and the dockerA/dockerB → MG communication paths. Investigate MG01/MG02 services and their upstream container interactions (network, processing queues, and service-side latencies) as the primary focus.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                        trace_name   timestamp                   ts_utc8     value        P95        P5     ratio   ratio_low
49        MG01  trace.from_dockerB1.duration_p95  1615291200 2021-03-09 12:00:00+08:00   0.88640   0.202547  0.008000  4.376257  110.800000
223   dockerB1        trace.to_MG01.duration_p95  1615291200 2021-03-09 12:00:00+08:00   0.88640   0.202547  0.008000  4.376257  110.800000
4309  dockerA2           trace.self.duration_p95  1615292280 2021-03-09 12:18:00+08:00   1.00480   0.301982  0.008450  3.327345  118.911243
5350      MG02  trace.from_dockerB2.duration_p95  1615292580 2021-03-09 12:23:00+08:00   0.51350   0.265212  0.007917  1.936183   64.856331
5485  dockerB2        trace.to_MG02.duration_p95  1615292580 2021-03-09 12:23:00+08:00   0.51350   0.265212  0.007917  1.936183   64.856331
1654      MG01  trace.from_dockerA1.duration_p95  1615291620 2021-03-09 12:07:00+08:00   0.61040   0.316297  0.007000  1.929829   87.200000
1789  dockerA1        trace.to_MG01.duration_p95  1615291620 2021-03-09 12:07:00+08:00   0.61040   0.316297  0.007000  1.929829   87.200000
4429      MG02  trace.from_dockerA1.duration_p95  1615292340 2021-03-09 12:19:00+08:00   1.01300   0.575233  0.008000  1.761027  126.625000
4528  dockerA1        trace.to_MG02.duration_p95  1615292340 2021-03-09 12:19:00+08:00   1.01300   0.575233  0.008000  1.761027  126.625000
2837      MG02     trace.from_dockerA2.row_count  1615291920 2021-03-09 12:12:00+08:00  69.00000  40.750000  8.000000  1.693252    8.625000
2948  dockerA2           trace.to_MG02.row_count  1615291920 2021-03-09 12:12:00+08:00  69.00000  40.750000  8.000000  1.693252    8.625000
4849      MG01  trace.from_dockerA2.duration_p95  1615292460 2021-03-09 12:21:00+08:00   0.91300   0.540317  0.007645  1.689747  119.424460
4996  dockerA2        trace.to_MG01.duration_p95  1615292460 2021-03-09 12:21:00+08:00   0.91300   0.540317  0.007645  1.689747  119.424460
5344      MG02  trace.from_dockerA2.duration_p95  1615292580 2021-03-09 12:23:00+08:00   1.37660   0.829592  0.007890  1.659369  174.474018
5455  dockerA2        trace.to_MG02.duration_p95  1615292580 2021-03-09 12:23:00+08:00   1.37660   0.829592  0.007890  1.659369  174.474018
2575      MG01  trace.from_dockerB2.duration_p95  1615291860 2021-03-09 12:11:00+08:00   0.14425   0.092265  0.008000  1.563431   18.031250
2746  dockerB2        trace.to_MG01.duration_p95  1615291860 2021-03-09 12:11:00+08:00   0.14425   0.092265  0.008000  1.563431   18.031250
3721      MG01    trace.to_dockerA1.duration_p95  1615292160 2021-03-09 12:16:00+08:00   1.66470   1.075572  0.324665  1.547734    5.127439
3832  dockerA1      trace.from_MG01.duration_p95  1615292160 2021-03-09 12:16:00+08:00   1.66470   1.075572  0.324665  1.547734    5.127439
1756  Tomcat03           trace.self.duration_p95  1615291620 2021-03-09 12:07:00+08:00   0.07065   0.047555  0.029230  1.485648    2.417037,      cmdb_id  high_anomaly_count  low_anomaly_count earliest_anomaly_timestamp
2       MG01                  81                 94  2021-03-09 12:00:00+08:00
3       MG02                  78                 94  2021-03-09 12:00:00+08:00
10  dockerB1                  34                 47  2021-03-09 12:00:00+08:00
8   dockerA1                  30                 44  2021-03-09 12:00:00+08:00
11  dockerB2                  32                 38  2021-03-09 12:00:00+08:00
7   Tomcat04                  35                 33  2021-03-09 12:00:00+08:00
5   Tomcat02                  33                 34  2021-03-09 12:00:00+08:00
9   dockerA2                  33                 34  2021-03-09 12:00:00+08:00
4   Tomcat01                  33                 33  2021-03-09 12:00:00+08:00
0       IG01                  30                 31  2021-03-09 12:01:00+08:00)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables
try:
    metric_anomalies_df
    metric_window_df
    metric_summary
except NameError:
    raise RuntimeError("Required kernel variables metric_anomalies_df, metric_window_df, or metric_summary are missing.")

# Candidate components to analyze
candidates = [
    "Mysql01","Mysql02","Redis01","Redis02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02"
]

# Filter anomalies to candidate components and ensure needed columns exist
df = metric_anomalies_df[metric_anomalies_df['cmdb_id'].isin(candidates)].copy()
# Keep relevant cols
df = df[['cmdb_id','kpi_name','timestamp','value','P95']].dropna(subset=['timestamp','value'])

# Assign consecutive-run ids per (cmdb_id, kpi_name) where consecutive minutes differ by exactly 60s
def assign_runs(g):
    g = g.sort_values('timestamp').reset_index(drop=True)
    g['ts_diff'] = g['timestamp'].diff().fillna(9999).astype(int)
    # start a new run when diff != 60
    g['new_run'] = (g['ts_diff'] != 60).astype(int)
    g['run_idx'] = g['new_run'].cumsum()  # run index within this group
    return g

df_runs = df.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_runs).reset_index(drop=True)

# Aggregate runs into fault records
agg = df_runs.groupby(['cmdb_id','kpi_name','run_idx'], as_index=False).agg(
    fault_start_ts_num=('timestamp','min'),
    fault_end_ts_num=('timestamp','max'),
    num_points=('timestamp','count'),
    max_value_in_run=('value','max'),
    P95=('P95','first')
)

# Compute durations (inclusive): if single point, duration 60s; else (end - start) + 60
agg['fault_duration_seconds'] = (agg['fault_end_ts_num'] - agg['fault_start_ts_num']) + 60
agg['max_ratio'] = agg.apply(
    lambda r: (np.inf if (r['P95']==0 and r['max_value_in_run']>0) else (1.0 if (r['P95']==0 and r['max_value_in_run']==0) else (r['max_value_in_run']/r['P95']))),
    axis=1
)

# Convert start/end to UTC+8 tz-aware timestamps for display
agg['fault_start_timestamp'] = pd.to_datetime(agg['fault_start_ts_num'], unit='s').dt.tz_localize(tz)
agg['fault_end_timestamp'] = pd.to_datetime(agg['fault_end_ts_num'], unit='s').dt.tz_localize(tz)

# Final faults table (store full table for later correlation)
faults_df = agg[[
    'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
    'num_points','max_value_in_run','P95','max_ratio','fault_duration_seconds',
    'fault_start_ts_num'
]].copy()

# 1) Top 20 faults sorted by max_ratio desc (compact columns)
faults_top20 = faults_df.sort_values('max_ratio', ascending=False).head(20)[
    ['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp','num_points','max_value_in_run','P95','max_ratio']
].reset_index(drop=True)

# 2) Aggregated fault summary per component: fault_count, longest_fault_duration_seconds, earliest_fault_start_timestamp
agg_comp = faults_df.groupby('cmdb_id', as_index=False).agg(
    fault_count=('kpi_name','count'),
    longest_fault_duration_seconds=('fault_duration_seconds','max'),
    earliest_fault_start_ts_num=('fault_start_ts_num','min')
)
agg_comp['earliest_fault_start_timestamp'] = pd.to_datetime(agg_comp['earliest_fault_start_ts_num'], unit='s').dt.tz_localize(tz)
faults_agg_top10 = agg_comp.sort_values('fault_count', ascending=False).head(10)[
    ['cmdb_id','fault_count','longest_fault_duration_seconds','earliest_fault_start_timestamp']
].reset_index(drop=True)

# Compact outputs
faults_top20, faults_agg_top10, faults_df.shape

```
Out[4]:
```
summary = (
    "Fault detection summary (candidate components only, incident window):\n\n"
    "- Total fault runs detected: 7,625.\n\n"
    "- Top components by number of faults (fault_count):\n"
    "  1) Mysql01: 1464\n"
    "  2) Mysql02: 1253\n"
    "  3) Redis01: 562\n"
    "  4) Tomcat02: 543\n"
    "  5) Redis02: 527\n"
    "  6) MG01: 496\n"
    "  7) Tomcat04: 479\n"
    "  8) Tomcat01: 469\n"
    "  9) IG02: 467\n"
    " 10) MG02: 460\n\n"
    "- Longest fault durations observed: up to 1,860 seconds (31 minutes) for several components; many earliest fault starts are at 2021-03-09 12:00:00+08.\n\n"
    "- Highest-severity individual faults (examples):\n"
    "  * Tomcat04 - OSLinux NETWORK TCP-CLOSE-WAIT (12:21–12:22) had P95==0 producing an infinite ratio (indicates previously unseen nonzero values).\n"
    "  * Mysql01/Mysql02 had several MySQL KPIs exceeding P95 by ~20x (e.g., Com Insert, LongestTrxActiveTime, CurrentSQLMaxRunningTime).\n"
    "  * MG01 also showed TCP-CLOSE-WAIT with ~20x exceedance.\n\n"
    "- Notes: Most faults are short runs (single-point anomalies are common), but their count is high for MySQL and Redis, indicating widespread repeated KPI exceedances across the window.\n\n"
    "Recommendation: prioritize investigation of Mysql01/Mysql02 first (high count and strong P95 exceedances), then Redis and Tomcat nodes, and MG/IG networking (TCP CLOSE-WAIT). Inspect MySQL transaction/handler metrics, Tomcat/MG TCP states, and local disk I/O spikes as likely contributing issues."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name     fault_start_timestamp       fault_end_timestamp  num_points  max_value_in_run       P95  max_ratio
0   Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 12:21:00+08:00 2021-03-09 12:22:00+08:00           2            1.0000  0.000000        inf
1    Mysql01                        Mysql-MySQL_3306_Com Insert 2021-03-09 12:30:00+08:00 2021-03-09 12:30:00+08:00           1            0.2667  0.013335  20.000000
2    Mysql02          Mysql-MySQL_3306_Opened table definitions 2021-03-09 12:26:00+08:00 2021-03-09 12:26:00+08:00           1            0.1667  0.008335  20.000000
3       MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 12:16:00+08:00 2021-03-09 12:16:00+08:00           1            1.0000  0.050000  20.000000
4    Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime 2021-03-09 12:04:00+08:00 2021-03-09 12:04:00+08:00           1            1.0000  0.050000  20.000000
5    Mysql02          Mysql-MySQL_3306_CurrentSQLMaxRunningTime 2021-03-09 12:04:00+08:00 2021-03-09 12:04:00+08:00           1            1.0000  0.050000  20.000000
6    Mysql02           Mysql-MySQL_3306_Table open cache misses 2021-03-09 12:26:00+08:00 2021-03-09 12:26:00+08:00           1            1.0333  0.130800   7.899847
7    Mysql02                     Mysql-MySQL_3306_Opened Tables 2021-03-09 12:26:00+08:00 2021-03-09 12:26:00+08:00           1            1.0333  0.130800   7.899847
8    Mysql02                  Mysql-MySQL_3306_Handler Read Rnd 2021-03-09 12:26:00+08:00 2021-03-09 12:26:00+08:00           1            0.2833  0.061665   4.594178
9    Mysql02                        Mysql-MySQL_3306_Sort Range 2021-03-09 12:26:00+08:00 2021-03-09 12:26:00+08:00           1            0.2833  0.061665   4.594178
10  Tomcat04                             OSLinux-CPU_CPU_CPUWio 2021-03-09 12:13:00+08:00 2021-03-09 12:13:00+08:00           1            1.8753  0.535700   3.500653
11   Redis02                             OSLinux-CPU_CPU_CPUWio 2021-03-09 12:23:00+08:00 2021-03-09 12:23:00+08:00           1            0.2835  0.083309   3.403014
12   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-09 12:14:00+08:00 2021-03-09 12:14:00+08:00           1            1.0000  0.300000   3.333333
13   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps 2021-03-09 12:12:00+08:00 2021-03-09 12:12:00+08:00           1            7.0000  2.100000   3.333333
14   Redis01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-09 12:21:00+08:00 2021-03-09 12:21:00+08:00           1            4.0000  1.200000   3.333333
15   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer... 2021-03-09 12:04:00+08:00 2021-03-09 12:04:00+08:00           1            0.4000  0.120000   3.333333
16   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps 2021-03-09 12:00:00+08:00 2021-03-09 12:00:00+08:00           1            0.2000  0.060000   3.333333
17   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead 2021-03-09 12:00:00+08:00 2021-03-09 12:00:00+08:00           1           22.4000  6.720000   3.333333
18   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-09 12:14:00+08:00 2021-03-09 12:14:00+08:00           1            4.0000  1.200000   3.333333
19   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead 2021-03-09 12:30:00+08:00 2021-03-09 12:30:00+08:00           1           22.4000  7.093310   3.157905,     cmdb_id  fault_count  longest_fault_duration_seconds earliest_fault_start_timestamp
0   Mysql01         1464                            1860      2021-03-09 12:00:00+08:00
1   Mysql02         1253                            1860      2021-03-09 12:00:00+08:00
2   Redis01          562                            1860      2021-03-09 12:00:00+08:00
3  Tomcat02          543                            1860      2021-03-09 12:00:00+08:00
4   Redis02          527                            1860      2021-03-09 12:00:00+08:00
5      MG01          496                            1860      2021-03-09 12:00:00+08:00
6  Tomcat04          479                            1860      2021-03-09 12:00:00+08:00
7  Tomcat01          469                            1860      2021-03-09 12:00:00+08:00
8      IG02          467                            1860      2021-03-09 12:00:00+08:00
9      MG02          460                            1860      2021-03-09 12:00:00+08:00, (7625, 10))```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse loaded dataframes when available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv')

# Incident window boundaries in UTC+8
start_ts = int(pd.Timestamp('2021-03-09 20:00:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-09 20:30:00', tz=tz).timestamp())

# Filter each file for rows inside the window (numeric timestamps in seconds)
metric_window_filtered = df_metric[df_metric['timestamp'].between(start_ts, end_ts)].copy()
trace_window_filtered  = df_trace[df_trace['timestamp'].between(start_ts, end_ts)].copy()
log_window_filtered    = df_log[df_log['timestamp'].between(start_ts, end_ts)].copy()
error_logs_window_filtered = df_err[df_err['timestamp'].between(start_ts, end_ts)].copy()

def compact_summary(df, name):
    row_count = df.shape[0]
    if row_count == 0:
        earliest = pd.NaT
        latest = pd.NaT
        cmdb_ids = []
    else:
        earliest = pd.to_datetime(df['timestamp'].min(), unit='s').tz_localize(tz)
        latest = pd.to_datetime(df['timestamp'].max(), unit='s').tz_localize(tz)
        cmdb_ids = list(pd.Series(df['cmdb_id'].dropna().unique())[:20])
    return pd.Series({
        'file': name,
        'row_count': row_count,
        'earliest_timestamp': earliest,
        'latest_timestamp': latest,
        'unique_cmdb_ids_sample': cmdb_ids
    })

summary_df = pd.DataFrame([
    compact_summary(metric_window_filtered, 'metric.csv'),
    compact_summary(trace_window_filtered,  'trace.csv'),
    compact_summary(log_window_filtered,    'log.csv'),
    compact_summary(error_logs_window_filtered, 'error_logs.csv')
])

# Explicit notes for files with zero rows (kept compact)
zero_row_files = summary_df[summary_df['row_count'] == 0]['file'].tolist()

# Display compact summary and shapes of filtered tables (stored for next steps)
summary_df[['file','row_count','earliest_timestamp','latest_timestamp','unique_cmdb_ids_sample']], \
metric_window_filtered.shape, trace_window_filtered.shape, log_window_filtered.shape, error_logs_window_filtered.shape, zero_row_files

```
Out[5]:
```
from datetime import datetime
tz_str = "UTC+8"

# Reuse summary_df from kernel and extract rows
s_metric = summary_df[summary_df['file']=='metric.csv'].iloc[0]
s_trace  = summary_df[summary_df['file']=='trace.csv'].iloc[0]
s_log    = summary_df[summary_df['file']=='log.csv'].iloc[0]
s_err    = summary_df[summary_df['file']=='error_logs.csv'].iloc[0]

def fmt_ids(id_list):
    if not id_list:
        return "[]"
    return "[" + ", ".join(id_list[:20]) + ("" if len(id_list) <= 20 else ", ...") + "]"

summary = (
    f"Data presence in window 2021-03-09 20:00:00 to 2021-03-09 20:30:00 ({tz_str}):\n\n"
    f"metric.csv: {int(s_metric['row_count'])} rows. Earliest: {s_metric['earliest_timestamp']}. Latest: {s_metric['latest_timestamp']}. "
    f"Sample cmdb_ids: {fmt_ids(s_metric['unique_cmdb_ids_sample'])}.\n\n"
    f"trace.csv: {int(s_trace['row_count'])} rows. Earliest: {s_trace['earliest_timestamp']}. Latest: {s_trace['latest_timestamp']}. "
    f"Sample cmdb_ids: {fmt_ids(s_trace['unique_cmdb_ids_sample'])}.\n\n"
    f"log.csv: {int(s_log['row_count'])} rows. Earliest: {s_log['earliest_timestamp']}. Latest: {s_log['latest_timestamp']}. "
    f"Sample cmdb_ids: {fmt_ids(s_log['unique_cmdb_ids_sample'])}.\n\n"
    f"error_logs.csv: {int(s_err['row_count'])} rows. No error log rows in the window.\n\n"
    f"Notes: Filtered tables have been stored in kernel variables (metric_window_filtered, trace_window_filtered, "
    f"log_window_filtered, error_logs_window_filtered) for further analysis."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count        earliest_timestamp          latest_timestamp                             unique_cmdb_ids_sample
0      metric.csv      27579 2021-03-09 12:00:00+08:00 2021-03-09 12:30:00+08:00  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...
1       trace.csv       6855 2021-03-09 12:00:00+08:00 2021-03-09 12:29:00+08:00  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...
2         log.csv        403 2021-03-09 12:00:00+08:00 2021-03-09 12:30:00+08:00  [Tomcat01, Tomcat02, Tomcat03, Tomcat04, apach...
3  error_logs.csv          0                       NaT                       NaT                                                 [], (27579, 5), (6855, 5), (403, 5), (0, 4), ['error_logs.csv'])```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse faults table from kernel (accept either faults_table or faults_df)
if 'faults_table' in globals():
    faults = globals()['faults_table']
elif 'faults_df' in globals():
    faults = globals()['faults_df']
else:
    raise RuntimeError("No faults table found in kernel (expected 'faults_table' or 'faults_df').")

# Candidate components
candidates = [
    "Mysql01","Mysql02","Redis01","Redis02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "MG01","MG02","IG01","IG02"
]

# Ensure timestamp column exists and is timezone-aware; convert if necessary
if 'fault_start_timestamp' in faults.columns:
    # If naive timestamps, localize to UTC+8
    if pd.api.types.is_datetime64_any_dtype(faults['fault_start_timestamp']):
        # check tzinfo of first non-null
        sample = faults['fault_start_timestamp'].dropna()
        if not sample.empty and sample.dt.tz is None:
            faults['fault_start_timestamp'] = faults['fault_start_timestamp'].dt.tz_localize(tz)
    else:
        faults['fault_start_timestamp'] = pd.to_datetime(faults['fault_start_timestamp'], utc=False).dt.tz_localize(tz)
else:
    raise RuntimeError("faults table missing 'fault_start_timestamp' column.")

# Filter to candidate components
filt = faults[faults['cmdb_id'].isin(candidates)].copy()

# Aggregation per component
agg = filt.groupby('cmdb_id', as_index=False).agg(
    fault_count=('kpi_name','count'),
    earliest_fault_start_timestamp=('fault_start_timestamp','min'),
    max_max_ratio=('max_ratio','max'),
    longest_fault_duration_seconds=('fault_duration_seconds','max')
)

# Ensure earliest timestamp is tz-aware and displayed in UTC+8
agg['earliest_fault_start_timestamp'] = pd.to_datetime(agg['earliest_fault_start_timestamp']).dt.tz_convert(tz)

# Sort by max_max_ratio desc (put inf first), then fault_count desc
# Use numpy to handle inf sorting naturally
agg_sorted = agg.sort_values(by=['max_max_ratio','fault_count'], ascending=[False, False]).reset_index(drop=True)

# Top 12 rows
agg_top12 = agg_sorted.head(12)

# Single row with overall highest max_max_ratio among candidates
if not agg_sorted.empty:
    top_component = agg_sorted.iloc[[0]]  # keep as DataFrame
else:
    top_component = agg_sorted.head(0)

# Compact outputs
agg_top12, top_component

```
Out[6]:
```
summary = (
    "Faults aggregation for candidate components (from stored faults table):\n\n"
    "Top 12 components (sorted by max_max_ratio then fault_count):\n"
    "1) Tomcat04 — fault_count=479, earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio=inf, longest_duration_s=1860\n"
    "2) Mysql01  — fault_count=1464, earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio=20.0, longest_duration_s=1860\n"
    "3) Mysql02  — fault_count=1253, earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio=20.0, longest_duration_s=1860\n"
    "4) MG01     — fault_count=496,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio=20.0, longest_duration_s=1860\n"
    "5) Redis02  — fault_count=527,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈3.403, longest_duration_s=1860\n"
    "6) Redis01  — fault_count=562,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈3.333, longest_duration_s=1860\n"
    "7) Tomcat03 — fault_count=457,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈1.837, longest_duration_s=1860\n"
    "8) IG01     — fault_count=448,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈1.833, longest_duration_s=1860\n"
    "9) MG02     — fault_count=460,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈1.818, longest_duration_s=1860\n"
    "10) Tomcat02— fault_count=543,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈1.818, longest_duration_s=1860\n"
    "11) Tomcat01— fault_count=469,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈1.667, longest_duration_s=1860\n"
    "12) IG02    — fault_count=467,  earliest_start=2021-03-09 12:00:00+08:00, max_max_ratio≈1.538, longest_duration_s=1860\n\n"
    "Overall highest max_max_ratio: Tomcat04 (max_max_ratio = inf) — indicates nonzero observed value vs a P95==0 for that KPI.\n\n"
    "Interpretation: prioritize investigation of Tomcat04 (infinite ratio case), then Mysql01/Mysql02 and MG01 (all have very large exceedances, ratio=20). Redis and Tomcat nodes also show notable but smaller max ratios. Many components have long reported fault durations (up to 1860s)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  fault_count earliest_fault_start_timestamp  max_max_ratio  longest_fault_duration_seconds
0   Tomcat04          479      2021-03-09 12:00:00+08:00            inf                            1860
1    Mysql01         1464      2021-03-09 12:00:00+08:00      20.000000                            1860
2    Mysql02         1253      2021-03-09 12:00:00+08:00      20.000000                            1860
3       MG01          496      2021-03-09 12:00:00+08:00      20.000000                            1860
4    Redis02          527      2021-03-09 12:00:00+08:00       3.403014                            1860
5    Redis01          562      2021-03-09 12:00:00+08:00       3.333333                            1860
6   Tomcat03          457      2021-03-09 12:00:00+08:00       1.836576                            1860
7       IG01          448      2021-03-09 12:00:00+08:00       1.832646                            1860
8       MG02          460      2021-03-09 12:00:00+08:00       1.818182                            1860
9   Tomcat02          543      2021-03-09 12:00:00+08:00       1.818182                            1860
10  Tomcat01          469      2021-03-09 12:00:00+08:00       1.666667                            1860
11      IG02          467      2021-03-09 12:00:00+08:00       1.538462                            1860,     cmdb_id  fault_count earliest_fault_start_timestamp  max_max_ratio  longest_fault_duration_seconds
0  Tomcat04          479      2021-03-09 12:00:00+08:00            inf                            1860)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse faults table from kernel
if 'faults_table' in globals():
    faults = globals()['faults_table']
elif 'faults_df' in globals():
    faults = globals()['faults_df']
elif 'faults' in globals():
    faults = globals()['faults']
else:
    raise RuntimeError("No faults table found in kernel (expected 'faults_table' or 'faults_df').")

# Ensure expected columns exist
expected_cols = {'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp','num_points','max_value_in_run','P95','max_ratio'}
missing = expected_cols - set(faults.columns)
if missing:
    raise RuntimeError(f"Faults table missing columns: {missing}")

# Candidate components
candidates = ["Tomcat04","Mysql01","Mysql02","MG01","Redis01","Redis02"]

# Ensure fault_start/end timestamps are timezone-aware in UTC+8
for col in ['fault_start_timestamp','fault_end_timestamp']:
    if not pd.api.types.is_datetime64_any_dtype(faults[col]):
        faults[col] = pd.to_datetime(faults[col], unit='s', errors='coerce')
    # localize if naive
    sample = faults[col].dropna()
    if not sample.empty and sample.dt.tz is None:
        faults[col] = faults[col].dt.tz_localize(tz)

# Filter to candidates
filt = faults[faults['cmdb_id'].isin(candidates)].copy()

# For each component pick the single fault record with largest max_ratio
# Sort by cmdb_id and descending max_ratio, then drop duplicates keeping first per cmdb_id
filt_sorted = filt.sort_values(['cmdb_id','max_ratio'], ascending=[True, False])
top_per_component = filt_sorted.drop_duplicates(subset=['cmdb_id'], keep='first')

# Now sort the selected rows by max_ratio descending and limit to 6
top_candidate_faults = top_per_component.sort_values('max_ratio', ascending=False).head(6)[
    ['cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp','num_points','max_value_in_run','P95','max_ratio']
].reset_index(drop=True)

# Display the result (variable name as required)
top_candidate_faults

```
Out[7]:
```
top = top_candidate_faults

lines = []
for _, r in top.iterrows():
    cm = r['cmdb_id']
    kpi = r['kpi_name']
    start = pd.to_datetime(r['fault_start_timestamp']).strftime('%Y-03-09 %H:%M:%S%z') if False else str(r['fault_start_timestamp'])
    # use str(...) to retain TZ display like +08:00
    start = str(r['fault_start_timestamp'])
    end = str(r['fault_end_timestamp'])
    pts = int(r['num_points'])
    mv = r['max_value_in_run']
    p95 = r['P95']
    mr = r['max_ratio']
    lines.append(f"{cm} — {kpi} | {start} to {end} | points={pts} | max={mv} | P95={p95} | ratio={mr}")

summary = (
    "Top fault per selected component (UTC+8):\n" +
    "\n".join(lines) +
    "\n\nKey takeaways:\n"
    "- Highest-severity fault: Tomcat04 (NETWORK TCP-CLOSE-WAIT) with infinite ratio (observed value vs P95==0).\n"
    "- Strong exceedances (ratio=20): MG01 (TCP CLOSE-WAIT), Mysql01 (Com Insert), Mysql02 (CurrentSQLMaxRunningTime).\n"
    "- Redis nodes show moderate peaks (max_ratio ≈ 3.4 and 3.33).\n\n"
    "Action: Prioritize investigating Tomcat04 and the MySQL/MG01 issues first, then check Redis resource/IO metrics."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                        kpi_name     fault_start_timestamp       fault_end_timestamp  num_points  max_value_in_run       P95  max_ratio
0  Tomcat04  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 12:21:00+08:00 2021-03-09 12:22:00+08:00           2            1.0000  0.000000        inf
1      MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 12:16:00+08:00 2021-03-09 12:16:00+08:00           1            1.0000  0.050000  20.000000
2   Mysql01                     Mysql-MySQL_3306_Com Insert 2021-03-09 12:30:00+08:00 2021-03-09 12:30:00+08:00           1            0.2667  0.013335  20.000000
3   Mysql02       Mysql-MySQL_3306_CurrentSQLMaxRunningTime 2021-03-09 12:04:00+08:00 2021-03-09 12:04:00+08:00           1            1.0000  0.050000  20.000000
4   Redis02                          OSLinux-CPU_CPU_CPUWio 2021-03-09 12:23:00+08:00 2021-03-09 12:23:00+08:00           1            0.2835  0.083309   3.403014
5   Redis01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-09 12:21:00+08:00 2021-03-09 12:21:00+08:00           1            4.0000  1.200000   3.333333```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables from kernel
try:
    trace_window_filtered
except NameError:
    trace_window_filtered = pd.read_csv('trace.csv')  # fallback (should exist)

# Ensure trace_summary exists; if not, compute global P5/P95 from full trace file (per rule 9)
if 'trace_summary' in globals():
    trace_summary = globals()['trace_summary']
else:
    # compute global thresholds from full trace data
    try:
        df_trace
    except NameError:
        df_trace = pd.read_csv('trace.csv')
    trace_summary = df_trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        P5=('value', lambda x: x.quantile(0.05)),
        P95=('value', lambda x: x.quantile(0.95))
    )

# 1) Select rows involving Tomcat04 (cmdb_id == 'Tomcat04' OR trace_name contains 'Tomcat04')
tomcat04_traces = trace_window_filtered[
    (trace_window_filtered['cmdb_id'] == 'Tomcat04') |
    (trace_window_filtered['trace_name'].str.contains('Tomcat04', na=False))
].copy()

# If empty, prepare empty outputs but keep variable defined
if tomcat04_traces.empty:
    top20_traces = tomcat04_traces.head(0)[['cmdb_id','trace_name','timestamp']]
    agg_summary = pd.DataFrame([{
        'total_rows': 0,
        'high_anomaly_count': 0,
        'low_anomaly_count': 0,
        'earliest_timestamp': pd.NaT,
        'latest_timestamp': pd.NaT
    }])
else:
    # 2) Join with trace_summary to get P95 and P5
    # Ensure trace_summary has P95 and P5 (if only P95/P5 names differ, adapt)
    if 'P95' not in trace_summary.columns or 'P5' not in trace_summary.columns:
        # attempt to compute if missing
        try:
            df_trace
        except NameError:
            df_trace = pd.read_csv('trace.csv')
        trace_summary = df_trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
            P5=('value', lambda x: x.quantile(0.05)),
            P95=('value', lambda x: x.quantile(0.95))
        )

    tomcat04_traces = tomcat04_traces.merge(
        trace_summary[['cmdb_id','trace_name','P95','P5']],
        on=['cmdb_id','trace_name'],
        how='left'
    )

    # human-readable tz-aware timestamp
    tomcat04_traces['ts_utc8'] = pd.to_datetime(tomcat04_traces['timestamp'], unit='s').dt.tz_localize(tz)

    # 3) Compute ratio_high and ratio_low with safe handling
    def compute_ratio_high(row):
        p95 = row['P95']
        val = row['value']
        if pd.isna(p95):
            return np.nan
        # per instruction: set to inf if P95==0
        if p95 == 0:
            return np.inf
        return val / p95

    def compute_ratio_low(row):
        p5 = row['P5']
        val = row['value']
        if pd.isna(p5):
            return np.nan
        if p5 == 0:
            # handle safely: if both zero, ratio_low = 1.0; else mark as inf to indicate non-comparable
            return 1.0 if val == 0 else np.inf
        return val / p5

    tomcat04_traces['ratio_high'] = tomcat04_traces.apply(compute_ratio_high, axis=1)
    tomcat04_traces['ratio_low'] = tomcat04_traces.apply(compute_ratio_low, axis=1)

    # Flags for anomalies (for aggregation)
    tomcat04_traces['is_high'] = tomcat04_traces['P95'].notna() & (tomcat04_traces['value'] >= tomcat04_traces['P95'])
    tomcat04_traces['is_low'] = tomcat04_traces['P5'].notna() & (tomcat04_traces['value'] <= tomcat04_traces['P5'])

    # 4A) Top 20 trace rows sorted by ratio_high descending
    top20_traces = tomcat04_traces.sort_values(by='ratio_high', ascending=False).head(20)[
        ['cmdb_id','trace_name','timestamp','ts_utc8','value','P95','P5','ratio_high','ratio_low']
    ].reset_index(drop=True)

    # 4B) Aggregated summary for Tomcat04
    total_rows = int(tomcat04_traces.shape[0])
    high_anomaly_count = int(tomcat04_traces['is_high'].sum())
    low_anomaly_count = int(tomcat04_traces['is_low'].sum())
    earliest_ts = pd.to_datetime(tomcat04_traces['timestamp'].min(), unit='s').tz_localize(tz)
    latest_ts = pd.to_datetime(tomcat04_traces['timestamp'].max(), unit='s').tz_localize(tz)

    agg_summary = pd.DataFrame([{
        'total_rows': total_rows,
        'high_anomaly_count': high_anomaly_count,
        'low_anomaly_count': low_anomaly_count,
        'earliest_timestamp': earliest_ts,
        'latest_timestamp': latest_ts
    }])

# Store tomcat04_traces in kernel (already assigned)
tomcat04_traces, top20_traces, agg_summary

```
Out[8]:
```
summary_lines = []

# Use existing kernel variables: tomcat04_traces, top20_traces, agg_summary
try:
    total_rows = int(agg_summary.at[0, 'total_rows'])
    high_count = int(agg_summary.at[0, 'high_anomaly_count'])
    low_count = int(agg_summary.at[0, 'low_anomaly_count'])
    earliest = str(agg_summary.at[0, 'earliest_timestamp'])
    latest = str(agg_summary.at[0, 'latest_timestamp'])
except Exception:
    total_rows = high_count = low_count = 0
    earliest = latest = "N/A"

summary_lines.append(
    f"Tomcat04 trace rows in window: {total_rows}. High anomalies (>=P95): {high_count}. Low anomalies (<=P5): {low_count}."
)
summary_lines.append(f"Earliest: {earliest}. Latest: {latest}.")

# Top examples by ratio_high from top20_traces (up to 3)
examples = []
if 'top20_traces' in globals() and not top20_traces.empty:
    for r in top20_traces.head(3).itertuples(index=False):
        examples.append(
            f"{r.cmdb_id} | {r.trace_name} | {r.ts_utc8} | value={r.value} P95={r.P95} ratio_high={r.ratio_high:.3f} ratio_low={r.ratio_low:.3f}"
        )
    summary_lines.append("Top trace exceedances (examples):")
    summary_lines.extend(examples)
else:
    summary_lines.append("No Tomcat04 trace rows found in the window.")

summary_lines.append(
    "Interpretation: Tomcat04-related traces show moderate high-side exceedances (ratio_high ~1.05–1.31 in top rows) "
    "and amplified ratio_low values where P5 is very small. Many anomalies are duration-related (duration_mean/duration_p95) "
    "and some row_count entries. Focus on Tomcat04 ↔ MG communication paths and duration spikes for investigation."
)

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp   cmdb_id                       trace_name       value                        ts         P95         P5                   ts_utc8  ratio_high  ratio_low  is_high  is_low
0    1615291200      IG01  trace.to_Tomcat04.duration_mean    0.282080 2021-03-09 12:00:00+08:00    0.341329   0.162946 2021-03-09 12:00:00+08:00    0.826417   1.731129    False   False
1    1615291200      IG01   trace.to_Tomcat04.duration_p95    1.083800 2021-03-09 12:00:00+08:00    2.056833   1.062695 2021-03-09 12:00:00+08:00    0.526927   1.019860    False   False
2    1615291200      IG01      trace.to_Tomcat04.row_count  125.000000 2021-03-09 12:00:00+08:00  133.550000  86.700000 2021-03-09 12:00:00+08:00    0.935979   1.441753    False   False
3    1615291200      IG02  trace.to_Tomcat04.duration_mean    0.254378 2021-03-09 12:00:00+08:00    0.322551   0.142250 2021-03-09 12:00:00+08:00    0.788645   1.788249    False   False
4    1615291200      IG02   trace.to_Tomcat04.duration_p95    1.153000 2021-03-09 12:00:00+08:00    2.002987   1.032852 2021-03-09 12:00:00+08:00    0.575640   1.116326    False   False
..          ...       ...                              ...         ...                       ...         ...        ...                       ...         ...        ...      ...     ...
808  1615292940  Tomcat04       trace.to_MG01.duration_p95    1.067900 2021-03-09 12:29:00+08:00    1.748477   1.032018 2021-03-09 12:29:00+08:00    0.610760   1.034769    False   False
809  1615292940  Tomcat04          trace.to_MG01.row_count   84.000000 2021-03-09 12:29:00+08:00  122.000000  82.900000 2021-03-09 12:29:00+08:00    0.688525   1.013269    False   False
810  1615292940  Tomcat04      trace.to_MG02.duration_mean    0.115962 2021-03-09 12:29:00+08:00    0.296434   0.107738 2021-03-09 12:29:00+08:00    0.391190   1.076336    False   False
811  1615292940  Tomcat04       trace.to_MG02.duration_p95    1.052200 2021-03-09 12:29:00+08:00    1.969695   0.972200 2021-03-09 12:29:00+08:00    0.534194   1.082288    False   False
812  1615292940  Tomcat04          trace.to_MG02.row_count   78.000000 2021-03-09 12:29:00+08:00  134.550000  85.900000 2021-03-09 12:29:00+08:00    0.579710   0.908033    False    True

[813 rows x 12 columns],      cmdb_id                         trace_name   timestamp                   ts_utc8       value         P95         P5  ratio_high  ratio_low
0   Tomcat04        trace.to_MG02.duration_mean  1615292460 2021-03-09 12:21:00+08:00    0.387793    0.296434   0.107738    1.308193   3.599418
1       MG02  trace.from_Tomcat04.duration_mean  1615292460 2021-03-09 12:21:00+08:00    0.387793    0.296434   0.107738    1.308193   3.599418
2       MG01  trace.from_Tomcat04.duration_mean  1615292160 2021-03-09 12:16:00+08:00    0.365655    0.285052   0.122505    1.282764   2.984818
3   Tomcat04        trace.to_MG01.duration_mean  1615292160 2021-03-09 12:16:00+08:00    0.365655    0.285052   0.122505    1.282764   2.984818
4   Tomcat04      trace.from_IG01.duration_mean  1615292160 2021-03-09 12:16:00+08:00    0.423020    0.341329   0.162946    1.239332   2.596080
5       IG01    trace.to_Tomcat04.duration_mean  1615292160 2021-03-09 12:16:00+08:00    0.423020    0.341329   0.162946    1.239332   2.596080
6   Tomcat04            trace.self.duration_p95  1615292160 2021-03-09 12:16:00+08:00    0.062800    0.051110   0.032270    1.228722   1.946080
7       IG02    trace.to_Tomcat04.duration_mean  1615291620 2021-03-09 12:07:00+08:00    0.382170    0.322551   0.142250    1.184837   2.686613
8   Tomcat04      trace.from_IG02.duration_mean  1615291620 2021-03-09 12:07:00+08:00    0.382170    0.322551   0.142250    1.184837   2.686613
9   Tomcat04         trace.to_MG01.duration_p95  1615292160 2021-03-09 12:16:00+08:00    2.042550    1.748477   1.032018    1.168188   1.979182
10      MG01   trace.from_Tomcat04.duration_p95  1615292160 2021-03-09 12:16:00+08:00    2.042550    1.748477   1.032018    1.168188   1.979182
11      MG02   trace.from_Tomcat04.duration_p95  1615292460 2021-03-09 12:21:00+08:00    2.077100    1.969695   0.972200    1.054529   2.136495
12  Tomcat04         trace.to_MG02.duration_p95  1615292460 2021-03-09 12:21:00+08:00    2.077100    1.969695   0.972200    1.054529   2.136495
13      MG01  trace.from_Tomcat04.duration_mean  1615292640 2021-03-09 12:24:00+08:00    0.298713    0.285052   0.122505    1.047923   2.438375
14  Tomcat04        trace.to_MG01.duration_mean  1615292640 2021-03-09 12:24:00+08:00    0.298713    0.285052   0.122505    1.047923   2.438375
15  Tomcat04          trace.from_IG02.row_count  1615291500 2021-03-09 12:05:00+08:00  147.000000  140.500000  85.600000    1.046263   1.717290
16      IG02        trace.to_Tomcat04.row_count  1615291500 2021-03-09 12:05:00+08:00  147.000000  140.500000  85.600000    1.046263   1.717290
17  Tomcat04           trace.self.duration_mean  1615292160 2021-03-09 12:16:00+08:00    0.049868    0.047732   0.024236    1.044759   2.057609
18      MG02   trace.from_Tomcat04.duration_p95  1615292220 2021-03-09 12:17:00+08:00    2.052000    1.969695   0.972200    1.041786   2.110677
19  Tomcat04         trace.to_MG02.duration_p95  1615292220 2021-03-09 12:17:00+08:00    2.052000    1.969695   0.972200    1.041786   2.110677,    total_rows  high_anomaly_count  low_anomaly_count        earliest_timestamp          latest_timestamp
0         813                  61                 57 2021-03-09 12:00:00+08:00 2021-03-09 12:29:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_window_filtered and trace_summary from kernel
try:
    trace_window_filtered
except NameError:
    trace_window_filtered = pd.read_csv('trace.csv')

try:
    trace_summary
except NameError:
    # compute global P5/P95 if not present
    df_trace = pd.read_csv('trace.csv')
    trace_summary = df_trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        P5=('value', lambda x: x.quantile(0.05)),
        P95=('value', lambda x: x.quantile(0.95))
    )

# Target critical timestamps in UTC+8 -> convert to numeric seconds
t1 = int(pd.Timestamp('2021-03-09 12:21:00', tz=tz).timestamp())
t2 = int(pd.Timestamp('2021-03-09 12:16:00', tz=tz).timestamp())
target_ts = {t1, t2}

# Filter rows at the exact timestamps
df_ts = trace_window_filtered[trace_window_filtered['timestamp'].isin(target_ts)].copy()

# Initial selection: rows directly involving Tomcat04
mask_initial = (
    (df_ts['cmdb_id'] == 'Tomcat04') |
    df_ts['trace_name'].str.contains('Tomcat04', na=False) |
    df_ts['trace_name'].str.contains('to_Tomcat04', na=False) |
    df_ts['trace_name'].str.contains('from_Tomcat04', na=False)
)
initial_selected = df_ts[mask_initial].copy()

# Derive tokens from trace names in initial selection (targets like MG01, dockerA1, etc.)
tokens = set()
for tn in initial_selected['trace_name'].dropna().unique():
    m_to = re.findall(r'trace\.to_([^\.]+)', tn)
    m_from = re.findall(r'trace\.from_([^\.]+)', tn)
    for x in m_to + m_from:
        tokens.add(x)
# Always include 'Tomcat04' token
tokens.add('Tomcat04')

# Build patterns to match related caller/callee trace rows at the same timestamps
patterns = []
for tok in tokens:
    patterns.append(fr'to_{re.escape(tok)}')
    patterns.append(fr'from_{re.escape(tok)}')
    patterns.append(re.escape(tok))
pattern_regex = "(" + "|".join(patterns) + ")"

# Final selection: any rows at these timestamps whose trace_name matches the pattern
tomcat04_trace_snap = df_ts[df_ts['trace_name'].str.contains(pattern_regex, na=False)].copy()

# Join with trace_summary to get global P95 and P5 (match on cmdb_id + trace_name)
tomcat04_trace_snap = tomcat04_trace_snap.merge(
    trace_summary[['cmdb_id','trace_name','P95','P5']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Compute ts_utc8, ratio_high, is_high, handling P95==0
tomcat04_trace_snap['ts_utc8'] = pd.to_datetime(tomcat04_trace_snap['timestamp'], unit='s').dt.tz_localize(tz)

def safe_ratio_high(val, p95):
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        return np.inf if val > 0 else 1.0
    return val / p95

tomcat04_trace_snap['ratio_high'] = tomcat04_trace_snap.apply(lambda r: safe_ratio_high(r['value'], r['P95']), axis=1)
tomcat04_trace_snap['is_high'] = tomcat04_trace_snap['P95'].notna() & (tomcat04_trace_snap['value'] >= tomcat04_trace_snap['P95'])

# Compact output 1: up to 50 rows sorted by timestamp asc then ratio_high desc
snap_out = tomcat04_trace_snap.sort_values(['timestamp','ratio_high'], ascending=[True, False])[
    ['timestamp','ts_utc8','cmdb_id','trace_name','value','P95','P5','ratio_high','is_high']
].head(50).reset_index(drop=True)

# Compact output 2: unique cmdb_id list and count of distinct trace_name values
unique_cmdbs = tomcat04_trace_snap['cmdb_id'].dropna().unique().tolist()
distinct_trace_names_count = int(tomcat04_trace_snap['trace_name'].nunique())

summary_row = pd.Series({
    'unique_cmdb_ids': unique_cmdbs,
    'num_distinct_trace_names': distinct_trace_names_count
})

# Store variable tomcat04_trace_snap in kernel (already assigned)
tomcat04_trace_snap, snap_out, summary_row

```
Out[9]:
```
Result: no matching trace rows were found.

- tomcat04_trace_snap is empty (no rows matched the exact timestamps 2021-03-09 12:21:00+08 or 2021-03-09 12:16:00+08 under the selection criteria).
- Unique cmdb_id list: [].
- Distinct trace_name count: 0.

Possible causes / next steps:
- The filtered trace table may not contain rows at the exact second values you requested (try using the exact epoch seconds 1615292460 and 1615292160 or expanding the search window by ±60s).
- Confirm the timestamps/timezone alignment in trace_window_filtered (UTC+8 was intended).
- If you want, I can re-run the selection using a ±1-minute tolerance around those timestamps or list nearby trace rows for Tomcat04 to capture caller/callee pairs. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value, ts, P95, P5, ts_utc8, ratio_high, is_high]
Index: [], Empty DataFrame
Columns: [timestamp, ts_utc8, cmdb_id, trace_name, value, P95, P5, ratio_high, is_high]
Index: [], unique_cmdb_ids             []
num_distinct_trace_names     0
dtype: object)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_window_filtered and trace_summary from kernel
# (they should already exist per previous steps)
# Define target timestamps (UTC+8) and convert to epoch seconds
t_targets = [
    int(pd.Timestamp('2021-03-09 12:21:00', tz=tz).timestamp()),
    int(pd.Timestamp('2021-03-09 12:16:00', tz=tz).timestamp())
]

# Build time mask for ±60 seconds around each target
time_mask = pd.Series(False, index=trace_window_filtered.index)
for t in t_targets:
    time_mask |= trace_window_filtered['timestamp'].between(t - 60, t + 60)

time_filtered = trace_window_filtered[time_mask].copy()

# Initial selection: rows explicitly referencing Tomcat04
init_mask = (
    (time_filtered['cmdb_id'] == 'Tomcat04') |
    time_filtered['trace_name'].str.contains('Tomcat04', na=False) |
    time_filtered['trace_name'].str.contains('to_Tomcat04', na=False) |
    time_filtered['trace_name'].str.contains('from_Tomcat04', na=False)
)
initial_sel = time_filtered[init_mask].copy()

# Extract tokens from trace_name (to_/from_ targets) to capture caller/callee pairs
tokens = set()
for tn in initial_sel['trace_name'].dropna().unique():
    tokens.update(re.findall(r'to_([^\.]+)', tn))
    tokens.update(re.findall(r'from_([^\.]+)', tn))
# Always include Tomcat04 token to match direct references
tokens.add('Tomcat04')

# Build regex pattern to match related traces (to_<token> or from_<token> or containing token)
pattern_parts = []
for tok in tokens:
    # match to_tok, from_tok, or tok directly (simple substring)
    pattern_parts.append(fr'to_{re.escape(tok)}')
    pattern_parts.append(fr'from_{re.escape(tok)}')
    pattern_parts.append(re.escape(tok))
pattern_regex = "(" + "|".join(pattern_parts) + ")"

# Final selection: any time-filtered rows that match the pattern OR are Tomcat04 by cmdb_id
final_mask = (
    time_filtered['trace_name'].str.contains(pattern_regex, na=False) |
    (time_filtered['cmdb_id'] == 'Tomcat04')
)
tomcat04_trace_snap = time_filtered[final_mask].copy()

# Ensure trace_summary has P95 and P5; compute if missing (use full trace series per rule 9)
if 'P95' not in trace_summary.columns or 'P5' not in trace_summary.columns:
    # compute global P5/P95 from df_trace if available
    if 'df_trace' in globals():
        df_full_trace = df_trace
    else:
        df_full_trace = pd.read_csv('trace.csv')
    trace_summary = df_full_trace.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        P5=('value', lambda x: x.quantile(0.05)),
        P95=('value', lambda x: x.quantile(0.95))
    )

# Join with global thresholds
if not tomcat04_trace_snap.empty:
    tomcat04_trace_snap = tomcat04_trace_snap.merge(
        trace_summary[['cmdb_id','trace_name','P95','P5']],
        on=['cmdb_id','trace_name'],
        how='left'
    )
    tomcat04_trace_snap['ts_utc8'] = pd.to_datetime(tomcat04_trace_snap['timestamp'], unit='s').dt.tz_localize(tz)

    def safe_ratio_high(val, p95):
        if pd.isna(p95):
            return np.nan
        if p95 == 0:
            return np.inf if val > 0 else 1.0
        return val / p95

    tomcat04_trace_snap['ratio_high'] = tomcat04_trace_snap.apply(lambda r: safe_ratio_high(r['value'], r['P95']), axis=1)
    tomcat04_trace_snap['is_high'] = tomcat04_trace_snap['P95'].notna() & (tomcat04_trace_snap['value'] >= tomcat04_trace_snap['P95'])
else:
    # keep an empty DataFrame with expected columns
    cols = list(trace_window_filtered.columns) + ['P95','P5','ts_utc8','ratio_high','is_high']
    tomcat04_trace_snap = pd.DataFrame(columns=cols)

# 1) Up to 100 rows sorted by timestamp asc then ratio_high desc with selected columns
snap_out = tomcat04_trace_snap.sort_values(['timestamp','ratio_high'], ascending=[True, False])[
    ['timestamp','ts_utc8','cmdb_id','trace_name','value','P95','P5','ratio_high','is_high']
].head(100).reset_index(drop=True)

# 2) Compact list of unique cmdb_id values and count of distinct trace_name values
unique_cmdbs = tomcat04_trace_snap['cmdb_id'].dropna().unique()
num_distinct_trace_names = int(tomcat04_trace_snap['trace_name'].nunique())

summary_row = pd.Series({
    'unique_cmdb_ids': list(unique_cmdbs)[:50],  # keep list compact
    'num_distinct_trace_names': num_distinct_trace_names
})

# Store tomcat04_trace_snap in kernel and return compact outputs
tomcat04_trace_snap, snap_out, summary_row

```
Out[10]:
```
summary = (
    "Result: No trace rows matched the requested selection.\n\n"
    "- tomcat04_trace_snap is empty: no rows found within ±60 seconds of 2021-03-09 12:21:00+08 or 2021-03-09 12:16:00+08 that match cmdb_id=='Tomcat04' or trace_name patterns ('Tomcat04', 'to_Tomcat04', 'from_Tomcat04').\n"
    "- Unique cmdb_id values in the selection: []\n"
    "- Distinct trace_name count: 0\n\n"
    "Possible reasons:\n"
    "  1) There are genuinely no trace records at those exact times (even within ±60s) for Tomcat04 or its caller/callee traces.\n"
    "  2) Trace naming conventions differ from the patterns used, so related caller/callee rows were not matched.\n"
    "  3) Timestamp alignment/timezone or window differences may mean the relevant rows are outside the inspected window.\n\n"
    "Recommended next steps (select one):\n"
    "  A) Expand the tolerance (e.g., ±300s) around each timestamp and re-run the search to catch near misses.\n"
    "  B) Search the entire trace dataset for any trace_name containing 'Tomcat04' to locate occurrences and their timestamps.\n"
    "  C) List trace_name prefixes around Tomcat04 (e.g., 'to_', 'from_') across nearby times to identify caller/callee pairs.\n\n"
    "Tell me which next step you prefer and I will run it."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value, ts, P95, P5, ts_utc8, ratio_high, is_high]
Index: [], Empty DataFrame
Columns: [timestamp, ts_utc8, cmdb_id, trace_name, value, P95, P5, ratio_high, is_high]
Index: [], unique_cmdb_ids             []
num_distinct_trace_names     0
dtype: object)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse existing dataframes if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# Incident window boundaries in UTC+8 (as numeric seconds)
start_ts = int(pd.Timestamp('2021-03-09 20:00:00', tz=tz).timestamp())
end_ts   = int(pd.Timestamp('2021-03-09 20:30:00', tz=tz).timestamp())

def file_summary(df, filename):
    row_count = int(df.shape[0])
    if row_count == 0:
        earliest_ts = pd.NaT
        latest_ts = pd.NaT
        earliest_num = None
        latest_num = None
        rows_in_window = 0
        window_covered = False
    else:
        earliest_num = int(df['timestamp'].min())
        latest_num = int(df['timestamp'].max())
        earliest_ts = pd.to_datetime(earliest_num, unit='s').tz_localize(tz)
        latest_ts = pd.to_datetime(latest_num, unit='s').tz_localize(tz)
        rows_in_window = int(df['timestamp'].between(start_ts, end_ts).sum())
        window_covered = (earliest_num <= start_ts) and (latest_num >= end_ts)
    return {
        'file': filename,
        'row_count': row_count,
        'earliest_timestamp': earliest_ts,
        'latest_timestamp': latest_ts,
        'rows_in_incident_window': rows_in_window,
        'window_covered': window_covered
    }

summaries = [
    file_summary(df_metric, 'metric.csv'),
    file_summary(df_trace,  'trace.csv'),
    file_summary(df_log,    'log.csv'),
    file_summary(df_err,    'error_logs.csv')
]

file_time_coverage = pd.DataFrame(summaries, columns=[
    'file','row_count','earliest_timestamp','latest_timestamp',
    'rows_in_incident_window','window_covered'
])

file_time_coverage

```
Out[11]:
```
Summary of full-file time coverage (incident window = 2021-03-09 20:00:00 to 2021-03-09 20:30:00 UTC+8):

- metric.csv: 27,579 rows total. Earliest = 2021-03-09 12:00:00+08:00, Latest = 2021-03-09 12:30:00+08:00. Rows inside incident window = 27,579. window_covered = True (the incident window is fully within this file's time range).

- trace.csv: 6,855 rows total. Earliest = 2021-03-09 12:00:00+08:00, Latest = 2021-03-09 12:29:00+08:00. Rows inside incident window = 6,855. window_covered = False (trace data stops at 12:29:00, so the incident window end 12:30:00 is not covered).

- log.csv: 403 rows total. Earliest = 2021-03-09 12:00:00+08:00, Latest = 2021-03-09 12:30:00+08:00. Rows inside incident window = 403. window_covered = True.

- error_logs.csv: 0 rows. No timestamps present; rows inside incident window = 0. window_covered = False.

Implication: metric and log data fully cover the incident window and can be used for complete window analysis. Trace data is available but does not include the final minute (12:29→12:30), and error_logs has no entries for this period.

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count        earliest_timestamp          latest_timestamp  rows_in_incident_window  window_covered
0      metric.csv      27579 2021-03-09 12:00:00+08:00 2021-03-09 12:30:00+08:00                    27579            True
1       trace.csv       6855 2021-03-09 12:00:00+08:00 2021-03-09 12:29:00+08:00                     6855           False
2         log.csv        403 2021-03-09 12:00:00+08:00 2021-03-09 12:30:00+08:00                      403            True
3  error_logs.csv          0                       NaT                       NaT                        0           False```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables from kernel
try:
    log_window_filtered
except NameError:
    log_window_filtered = pd.read_csv('log.csv')

try:
    log_summary
except NameError:
    # compute global log_summary (P95) if not present
    df_log = pd.read_csv('log.csv')
    log_summary = df_log.groupby(['cmdb_id','log_name'], as_index=False).agg(
        count=('value','count'),
        min=('value','min'),
        P50=('value', lambda x: x.quantile(0.5)),
        P95=('value', lambda x: x.quantile(0.95)),
        max=('value','max')
    )

# Candidate components
candidates = ["Tomcat04","Mysql01","Mysql02","MG01","Redis01","Redis02"]

# 1) Filter log_window_filtered to candidate components
candidate_logs = log_window_filtered[log_window_filtered['cmdb_id'].isin(candidates)].copy()

# 2) Join with log_summary to obtain group's global P95
candidate_logs = candidate_logs.merge(
    log_summary[['cmdb_id','log_name','P95']],
    on=['cmdb_id','log_name'],
    how='left'
)

# Add tz-aware timestamp column for display
candidate_logs['ts_utc8'] = pd.to_datetime(candidate_logs['timestamp'], unit='s').dt.tz_localize(tz)

# 3) Identify log anomaly rows where value >= P95; compute ratio with safe handling
def compute_ratio(val, p95):
    if pd.isna(p95):
        return np.nan
    if p95 == 0:
        return np.inf if val > 0 else 1.0
    return val / p95

candidate_logs['ratio'] = candidate_logs.apply(lambda r: compute_ratio(r['value'], r['P95']), axis=1)
candidate_logs['is_anomaly'] = candidate_logs['P95'].notna() & (candidate_logs['value'] >= candidate_logs['P95'])

# Full anomalies table (stored)
candidate_log_anomalies = candidate_logs[candidate_logs['is_anomaly']].copy()

# 4) Top 20 anomaly rows sorted by ratio desc (compact selection)
candidate_log_anomalies_top20 = candidate_log_anomalies.sort_values('ratio', ascending=False)[
    ['cmdb_id','log_name','timestamp','ts_utc8','value','P95','ratio']
].head(20).reset_index(drop=True)

# 5) Aggregated table per candidate component
agg = candidate_log_anomalies.groupby('cmdb_id').agg(
    anomaly_count=('is_anomaly','sum'),
    earliest_anomaly_ts_num=('timestamp','min'),
    distinct_log_names=('log_name', lambda s: sorted(s.dropna().unique()))
).reset_index()

# Limit distinct log_name lists to first 10 entries for compactness
agg['distinct_log_names_limited'] = agg['distinct_log_names'].apply(lambda lst: lst[:10])

# Convert earliest timestamp to tz-aware display
agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anomaly_ts_num'], unit='s').dt.tz_localize(tz)

# Select and sort top 10 components by anomaly_count desc
candidate_log_agg_top10 = agg.sort_values('anomaly_count', ascending=False).head(10)[
    ['cmdb_id','anomaly_count','earliest_anomaly_timestamp','distinct_log_names_limited']
].reset_index(drop=True)

# Store full tables in kernel variables (already assigned): candidate_logs, candidate_log_anomalies

# Compact outputs
candidate_log_anomalies_top20, candidate_log_agg_top10

```
Out[12]:
```
Summary of log analysis for candidate components (Tomcat04, Mysql01, Mysql02, MG01, Redis01, Redis02):

- Findings: only Tomcat04 produced log anomalies in the incident window.
  - Total anomaly rows for Tomcat04: 35 (earliest anomaly at 2021-03-09 12:00:00+08).
  - Distinct log names that triggered anomalies (sample): log.error_count, log.gc_pause_max, log.row_count.

- Top anomaly examples (highest ratios):
  - Tomcat04 — log.row_count at 2021-03-09 12:24:00+08: value=259 vs P95=254.5 (ratio ≈ 1.018).
  - Tomcat04 — log.row_count at 2021-03-09 12:12:00+08: value=255 vs P95=254.5 (ratio ≈ 1.002).
  - Many log.error_count rows are equal to their global P95 (P95==0 and value==0), which were flagged as anomalies by the applied rule but likely indicate no errors rather than meaningful spikes.
  - log.gc_pause_max entries hit their P95 (e.g., 0.08s) but show only borderline exceedances.

- Interpretation / recommendations:
  - Prioritize investigating Tomcat04: modest spikes in log row counts and occasional GC pause maxima coincide with the detected faults/trace signals.
  - Treat log.error_count "anomalies" with caution when both value and P95 are zero — these are not actionable errors by themselves.
  - Correlate the log.row_count and gc_pause_max timestamps (notably around 12:12 and 12:24) with metric and trace faults on Tomcat04 for root-cause tracing.

Stored variables: candidate_logs (filtered candidate log rows) and candidate_log_anomalies (detected anomalies) are available in the kernel for further inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name   timestamp                   ts_utc8   value     P95     ratio
0   Tomcat04     log.row_count  1615292640 2021-03-09 12:24:00+08:00  259.00  254.50  1.017682
1   Tomcat04     log.row_count  1615291920 2021-03-09 12:12:00+08:00  255.00  254.50  1.001965
2   Tomcat04   log.error_count  1615291200 2021-03-09 12:00:00+08:00    0.00    0.00  1.000000
3   Tomcat04   log.error_count  1615291320 2021-03-09 12:02:00+08:00    0.00    0.00  1.000000
4   Tomcat04   log.error_count  1615291260 2021-03-09 12:01:00+08:00    0.00    0.00  1.000000
5   Tomcat04   log.error_count  1615291440 2021-03-09 12:04:00+08:00    0.00    0.00  1.000000
6   Tomcat04   log.error_count  1615291380 2021-03-09 12:03:00+08:00    0.00    0.00  1.000000
7   Tomcat04   log.error_count  1615291560 2021-03-09 12:06:00+08:00    0.00    0.00  1.000000
8   Tomcat04   log.error_count  1615291620 2021-03-09 12:07:00+08:00    0.00    0.00  1.000000
9   Tomcat04   log.error_count  1615291680 2021-03-09 12:08:00+08:00    0.00    0.00  1.000000
10  Tomcat04  log.gc_pause_max  1615291380 2021-03-09 12:03:00+08:00    0.08    0.08  1.000000
11  Tomcat04   log.error_count  1615291740 2021-03-09 12:09:00+08:00    0.00    0.00  1.000000
12  Tomcat04   log.error_count  1615291800 2021-03-09 12:10:00+08:00    0.00    0.00  1.000000
13  Tomcat04   log.error_count  1615291860 2021-03-09 12:11:00+08:00    0.00    0.00  1.000000
14  Tomcat04  log.gc_pause_max  1615291800 2021-03-09 12:10:00+08:00    0.08    0.08  1.000000
15  Tomcat04   log.error_count  1615291920 2021-03-09 12:12:00+08:00    0.00    0.00  1.000000
16  Tomcat04   log.error_count  1615291980 2021-03-09 12:13:00+08:00    0.00    0.00  1.000000
17  Tomcat04   log.error_count  1615292040 2021-03-09 12:14:00+08:00    0.00    0.00  1.000000
18  Tomcat04   log.error_count  1615291500 2021-03-09 12:05:00+08:00    0.00    0.00  1.000000
19  Tomcat04   log.error_count  1615292100 2021-03-09 12:15:00+08:00    0.00    0.00  1.000000,     cmdb_id  anomaly_count earliest_anomaly_timestamp                         distinct_log_names_limited
0  Tomcat04             35  2021-03-09 12:00:00+08:00  [log.error_count, log.gc_pause_max, log.row_co...)```
```